In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from pathlib import Path
import time
import datetime as dt

user = "wconceicao"
password = "zJm7$j%qRU@WoCxM"
host = "dw-ro.data.grupoa.education"
port = "5432"
database = "postgres"
password_encoded = quote_plus(password)
engine = create_engine(
    f"postgresql+psycopg2://{user}:{password_encoded}@{host}:{port}/{database}"
    )

In [2]:
def executar_qry(query,nome_df):

    inicio = time.time()

    try:
        print(f'Executando: {nome_df}...')
        df = pd.read_sql(text(query),engine)

        tempo_total = time.time() - inicio
        minutos = int(tempo_total // 60)
        segundos = tempo_total % 60

        if minutos > 0:
            tempo = f'{minutos}min {segundos:.1f}s'
        else:
            tempo = f'{segundos:.2f}s' 

        num_linhas = len(df)

        print(f'✅ {nome_df}: {num_linhas} linhas | ⏱️{tempo}')

        return df       

    except Exception as e:
        tempo_total = time.time() - inicio
        print(f' Erro em {nome_df} após {tempo_total:.2f}s: {e}')
        return None

In [3]:
qry_sispag = """
    with Sispag AS (
SELECT
    CASE
        WHEN tipoproduto = 'C' THEN 'ARTMED360'
        ELSE 'SECAD'
    END AS "nome_ies",

    RIGHT(REPLACE(REPLACE(REPLACE(REPLACE(app_sispag_pagamento.vd_cliente.celular, '(', ''), ')', ''), '-', ''), ' ', ''), 11) AS "celular",



    vd_produto.tipoproduto::text,

    CASE
        WHEN vd_compra."codVendedor" IS NULL THEN DATE(timezone('America/Sao_Paulo'::text, timezone('UTC'::text, vd_compra.datahora)))
        WHEN vd_compra."codVendedor"::text = '8000'::text THEN DATE(timezone('America/Sao_Paulo'::text, timezone('UTC'::text, vd_compra.datahora)))
        ELSE DATE(vd_compra.datahora)
        
    END AS dt,


    TO_CHAR(DATE_TRUNC('hour', vd_compra.datahora), 'HH24:MI') AS hr,

    vd_compra.compraid AS id,
    vd_compra.clienteid AS user_id,
    
    vd_cliente.nome AS name,
    LOWER(vd_cliente.email::text) AS email,
    vd_cliente.cidade AS city,
    vd_cliente."UF" AS state,
    vd_compraitem.produtoid AS product_id,

    CASE
        WHEN vd_produto.nomeresumido::text = 'PROENDÓCRINO'::text THEN 'PROENDOCRINO'::character varying
        WHEN vd_produto.nomeresumido::text = 'PROENF-URG'::text THEN 'PROENF/URG'::character varying
        WHEN vd_produto.nomeresumido::text = 'PROFISIO/NEURO'::text THEN 'PROFISIO/NEF'::character varying
        WHEN vd_produto.nomeresumido::text = 'PROFISIO/TRAUMA'::text THEN 'PROFISIO/TO'::character varying
        WHEN vd_produto.nomeresumido::text = 'PROMEVET'::text THEN 'PROMEVET/PA'::character varying
        WHEN vd_produto.nomeresumido::text = 'PROTERAPÊUTICA'::text THEN 'PROTERAPEUTICA'::character varying
        WHEN vd_produto.nomeresumido::text = 'PROURGEM'::text THEN 'PROURGEN'::character varying
        WHEN vd_produto.nomeresumido::text = 'PRO-UROLOGIA'::text THEN 'PROUROLOGIA'::character varying
        ELSE vd_produto.nomeresumido
    END AS program,

    CASE
        WHEN programs.area IS NOT NULL THEN programs.area
        ELSE
            CASE
                WHEN vd_produto.nomeresumido::text = ANY (ARRAY['PROURGEM'::character varying::text, 'PROENDÓCRINO'::character varying::text, 'PRO-UROLOGIA'::character varying::text, 'PROTERAPÊUTICA'::character varying::text]) THEN 'Medicina'::text
                WHEN vd_produto.nomeresumido::text = 'PROMEVET'::text THEN 'Veterinária'::text
                WHEN vd_produto.nomeresumido::text = 'PROENF-URG'::text THEN 'Enfermagem'::text
                WHEN vd_produto.nomeresumido::text = ANY (ARRAY['PROFISIO/NEURO'::character varying::text, 'PROFISIO/TRAUMA'::character varying::text]) THEN 'Fisioterapia'::text
                ELSE NULL::text
            END
    END AS area,

    vd_compraitem.valor * vd_compra.valortotal / SUM(vd_compraitem.valor) OVER (PARTITION BY vd_compra.compraid) AS value,

    -- CORREÇÃO AQUI: Removido o 'AS' duplicado
    CASE 
        WHEN vd_compra.formapagamento = 'A' THEN 'A Vista'
        WHEN vd_compra.formapagamento = 'C' THEN 'Credito'
        WHEN vd_compra.formapagamento = 'D' THEN 'Debito'
        WHEN vd_compra.formapagamento = 'P' THEN 'Pix'
        WHEN vd_compra.formapagamento = 'R' THEN 'Recorrente'
        ELSE 'Not_found'       
    END AS payment_type,

    vd_compra."codVendedor" AS channel_id,

    CASE
        WHEN vd_compra."codVendedor" IS NULL THEN 'eCommerce'
        WHEN LEFT(vd_compra."codVendedor"::text, 1) = '8' THEN 'Representantes'
        WHEN vd_compra."codVendedor"::text IN ('9186', '9323', '9326', '9325') THEN 'Receptivo'
        WHEN LEFT(vd_compra."codVendedor"::text, 1) = 'R' THEN 'Renovacao'
        ELSE 'Call Center'
    END AS channel

FROM app_sispag_pagamento.vd_compra
LEFT JOIN app_sispag_pagamento.vd_compraitem ON vd_compra.compraid = vd_compraitem.compraid
LEFT JOIN app_sispag_pagamento.vd_produto ON vd_compraitem.produtoid = vd_produto.produtoid
LEFT JOIN app_sispag_pagamento.vd_request ON vd_compra.requestid = vd_request.requestid
LEFT JOIN app_sispag_pagamento.vd_cliente ON vd_compra.clienteid = vd_cliente.clienteid
LEFT JOIN bu_secad.programs ON vd_produto.nomeresumido::text = programs.program

WHERE DATE(timezone('America/Sao_Paulo'::text, timezone('UTC'::text, vd_compra.datahora))) >= '2025-10-01'::date
  AND vd_produto.tipoproduto::text IN ('P', 'C')
  AND vd_request.ambiente::text = 'P'::text
  AND (vd_compra."codVendedor"::text <> '123'::text OR vd_compra."codVendedor" IS NULL)
  AND LOWER(vd_cliente.nome::text) NOT LIKE '%teste%'
)

select *    
FROM Sispag  
WHERE channel NOT IN ('eCommerce','Representantes')
and tipoproduto = 'P'
"""


df = {}

try:
    queries = {
        'sispag': qry_sispag,
        
    }
    for nome, qry in queries.items():
        df[nome] = executar_qry(qry, nome)
        
except Exception as e:
    print(f'Erro geral na execução das queries: {e}')        



Executando: sispag...
✅ sispag: 1958 linhas | ⏱️4.22s


In [4]:
df_meta = pd.read_excel(r'C:\Users\wconceicao\OneDrive - Grupo A Educação SA\Área de Trabalho\Construtores\Meta Secad.xlsx')

In [5]:
meta_diaria = df_meta.copy()
meta_diaria['canal'] = None
meta_diaria.loc[
    meta_diaria['channel'].isin(['Call Center','Receptivo']),
    'canal'
] = 'Secad'
meta_diaria = meta_diaria[meta_diaria['canal'] == 'Secad']
meta_diaria = meta_diaria.groupby('date').agg(
    meta_dia = ('goal','sum')
).reset_index()
meta_diaria['meta_dia'] = meta_diaria['meta_dia'].round(0)

In [6]:
meta_qtd_mes = meta_diaria['meta_dia'].sum()
valor_meta_mes = 959381.00
valor_meta_70 = valor_meta_mes * 0.70
ticket_medio = (valor_meta_mes / meta_qtd_mes)
data_inicio = '2026-01-05'
data_fim = '2026-01-31'
hoje = pd.Timestamp.now().normalize()

In [7]:
meta_diaria['meta_valor_dia'] = (
    meta_diaria['meta_dia'] * ticket_medio
)
meta_diaria['meta_valor_dia'] = meta_diaria['meta_valor_dia'].round(2)
meta_diaria['date'] = pd.to_datetime(meta_diaria['date'])

In [8]:
sispag = df['sispag'].copy()
sispag['dt'] = pd.to_datetime(sispag['dt'])

sispag = (
    sispag
    .groupby(['dt','hr'])
    .agg(
        valor=('value','sum'),
        qtde=('id','count')
    )
    .reset_index()
)

realizado_dia = (
    sispag.groupby('dt').agg(
        vendas_real = ('qtde','sum'),
        vendas_valor = ('valor','sum')
    )
).reset_index()

meta_diaria = meta_diaria.merge(
    realizado_dia[['dt','vendas_real','vendas_valor']],
    left_on='date',
    right_on='dt',
    how='left'
)

In [9]:
meta_diaria['vendas_valor'] = meta_diaria['vendas_valor'].round(2)
meta_diaria = meta_diaria[['date','vendas_real','vendas_valor','meta_dia','meta_valor_dia']]
meta_diaria = meta_diaria.fillna(0)
meta_diaria['gap_vendas'] = meta_diaria['vendas_real'] - meta_diaria['meta_dia']
meta_diaria['gap_valor'] = meta_diaria['vendas_valor'] - meta_diaria['meta_valor_dia']
hoje = pd.Timestamp.today().normalize()
meta_diaria['dia_passado'] = meta_diaria['date'] <= hoje
meta_diaria['dia_futuro'] = meta_diaria['date'] > hoje
meta_diaria.loc[~meta_diaria['dia_passado'],['gap_vendas','gap_valor']] = 0



In [10]:
meta_qtd_mensal = meta_diaria['meta_dia'].sum()
qtd_realizada = meta_diaria.loc[meta_diaria['dia_passado'],'vendas_real'].sum()
qtd_faltante = meta_qtd_mensal - qtd_realizada

meta_valor_mensal = meta_diaria['meta_valor_dia'].sum()
valor_realizado = meta_diaria.loc[
    meta_diaria['dia_passado'], 'vendas_valor'
].sum()

valor_faltante = meta_valor_mensal - valor_realizado

dias_restantes = meta_diaria.loc[
    meta_diaria['dia_futuro']
].shape[0]

In [11]:
meta_diaria['meta_recalc_qtd'] = 0

meta_diaria.loc[
    meta_diaria['dia_futuro'],
    'meta_recalc_qtd'
] = qtd_faltante / dias_restantes

meta_diaria['meta_recalc_valor'] = 0

meta_diaria.loc[
    meta_diaria['dia_futuro'],
    'meta_recalc_valor'
] = valor_faltante / dias_restantes



C:\Users\wconceicao\AppData\Local\Temp\ipykernel_17684\4176441496.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '45.875' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  meta_diaria.loc[
C:\Users\wconceicao\AppData\Local\Temp\ipykernel_17684\4176441496.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '67475.40124999998' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  meta_diaria.loc[


In [12]:
meta_mensal_qtd = meta_diaria['meta_dia'].sum()
meta_mensal_valor = meta_diaria['meta_valor_dia'].sum()

In [13]:
realizado_acum_qtd = meta_diaria.loc[meta_diaria['dia_passado'],'vendas_real'].sum()
realizado_acum_valor = meta_diaria.loc[meta_diaria['dia_passado'],'vendas_valor'].sum()

In [14]:
meta_diaria

,date,vendas_real,vendas_valor,meta_dia,meta_valor_dia,gap_vendas,gap_valor,dia_passado,dia_futuro,meta_recalc_qtd,meta_recalc_valor
0,2026-01-01,0.0,0.00,0.0,0.00,0.0,0.00,True,False,0.000,0.00000
1,2026-01-02,0.0,0.00,0.0,0.00,0.0,0.00,True,False,0.000,0.00000
2,2026-01-03,0.0,0.00,0.0,0.00,0.0,0.00,True,False,0.000,0.00000
3,2026-01-04,0.0,0.00,0.0,0.00,0.0,0.00,True,False,0.000,0.00000
4,2026-01-05,14.0,20050.11,31.0,43930.30,-17.0,-23880.19,True,False,0.000,0.00000
5,2026-01-06,21.0,28846.72,37.0,52432.94,-16.0,-23586.22,True,False,0.000,0.00000
6,2026-01-07,27.0,47943.89,33.0,46764.51,-6.0,1179.38,True,False,0.000,0.00000
7,2026-01-08,16.0,21581.53,26.0,36844.77,-10.0,-15263.24,True,False,0.000,0.00000
8,2026-01-09,23.0,27650.62,31.0,43930.30,-8.0,-16279.68,True,False,0.000,0.00000
9,2026-01-10,14.0,18569.24,12.0,17005.28,2.0,1563.96,True,False,0.000,0.00000
